In [4]:
import pandas as pd
df = pd.read_csv('../data/processed/filtered_complaints.csv')

df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,...,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,narrative_word_count,has_narrative,cleaned_narrative
0,2025-06-13,Credit card,Store credit card,Getting a credit card,Card opened without my consent or knowledge,A XXXX XXXX card was opened under my name by a...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",TX,78230,...,Consent provided,Web,2025-06-13,Closed with non-monetary relief,Yes,NaN,14069121,91,True,a xxxx xxxx card was opened under my name by a...
1,2025-06-13,Savings account,Checking account,Managing an account,Deposits and withdrawals,I made the mistake of using my wellsfargo debi...,Company has responded to the consumer and the ...,WELLS FARGO & COMPANY,ID,83815,...,Consent provided,Web,2025-06-13,Closed with explanation,Yes,NaN,14061897,109,True,i made the mistake of using my wellsfargo debi...
2,2025-06-12,Credit card,General-purpose credit card or charge card,"Other features, terms, or problems",Other problem,"Dear CFPB, I have a secured credit card with c...",Company has responded to the consumer and the ...,"CITIBANK, N.A.",NY,11220,...,Consent provided,Web,2025-06-13,Closed with monetary relief,Yes,NaN,14047085,156,True,dear cfpb i have a secured credit card with ci...
3,2025-06-12,Credit card,General-purpose credit card or charge card,Incorrect information on your report,Account information incorrect,I have a Citi rewards cards. The credit balanc...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",IL,60067,...,Consent provided,Web,2025-06-12,Closed with explanation,Yes,NaN,14040217,233,True,i have a citi rewards cards the credit balance...
4,2025-06-09,Credit card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,b'I am writing to dispute the following charge...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",TX,78413,...,Consent provided,Web,2025-06-09,Closed with monetary relief,Yes,NaN,13968411,454,True,b i am writing to dispute the following charge...


In [5]:
from sklearn.model_selection import train_test_split

n_samples = 12000

df_sampled, _ = train_test_split(
    df,
    stratify=df['Product'],
    train_size = n_samples,
    random_state=42
)
df_sampled['Product'].value_counts(normalize=True)

Product
Credit card        0.394000
Savings account    0.323000
Money transfers    0.205333
Personal loan      0.077667
Name: proportion, dtype: float64

In [6]:
def chunk_text(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

example_chunks = chunk_text(df_sampled.iloc[0]['cleaned_narrative'], chunk_size=500, overlap=50)
print(example_chunks[:3])


['i want to cancel automatic payment because i am going to pay off the loan in a lump sum xxxx xxxx says i must call to cancel autopay i called the provided number they indicated that i am eligible for a 100 00 reward there is no option to skip this as every option to be connected to an operator says i must accept this reward when i was finally connected to an operator they stated i must provide my personal banking details so they can give me the reward they dont want my loan details just my personal banking information so they can give me a reward i let them know i dont want the reward and im not providing my bank details i just want to cancel automatic payment the operator got upset at me and demanded i provide personal bank details in order to proceed and receive the reward again they werent asking for my loan number to help me cancel the auto pay they wanted my bank information to give me money when i declined the operator hung up on me i then contacted the online chat and told the

In [7]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


all_chunks = []
metadata = []

for idx, row in df_sampled.iterrows():
    chunks = chunk_text(row['cleaned_narrative'], chunk_size=500, overlap=50)
    all_chunks.extend(chunks)
    metadata.extend([
        {
            'complaint_id':row['Complaint ID'], 
            'product':row['Product'],
            'text': chunk
            } 
        for chunk in chunks])
    
embeddings = model.encode(all_chunks, show_progress_bar=True)
embeddings = np.array(embeddings).astype('float32')

Batches: 100%|██████████| 419/419 [07:57<00:00,  1.14s/it]  


In [8]:
d = embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(embeddings)

faiss.write_index(index, '../vector_store/faiss_index.index')

import pickle
with open('../vector_store/metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

In [9]:
import numpy as np

query = "I have an issue with my credit card statement"
query_vec = model.encode([query]).astype("float32")

D, I = index.search(query_vec, k=3)

for idx, dist in zip(I[0], D[0]):
    print("Score:", dist)
    print("Product:", metadata[idx]['product'])
    print("Complaint ID:", metadata[idx]['complaint_id'])
    print("Chunk:", all_chunks[idx][:200], "...")  
    print("---")


Score: 0.54391134
Product: Credit card
Complaint ID: 2753405
Chunk: bank of america credit card xxxx xxxx xxxx xxxx refuses to send me my xx xx xxxx monthly statement i asked for this statement i never got this statement instead i got an irrelevant 92 page document it ...
---
Score: 0.55625564
Product: Credit card
Complaint ID: 7481156
Chunk: my statement of the situation with this credit card company is attached if you need anything else please call ...
---
Score: 0.65389407
Product: Credit card
Complaint ID: 1399576
Chunk: my xx xx xxxx statement arrived and upon review i had noticed charges that i did not make futhermore i only made xxxx purchase second i contacted the company and they stated they would dispute the ite ...
---


In [10]:
import pickle 
with open("../vector_store/metadata.pkl", "rb") as f:
          metadata = pickle.load(f)
print(metadata[0].keys())

dict_keys(['complaint_id', 'product', 'text'])


In [11]:
print(metadata[0]["text"][:150])

i want to cancel automatic payment because i am going to pay off the loan in a lump sum xxxx xxxx says i must call to cancel autopay i called the prov
